# 실기 대비
# 실전 문제풀이
# set 5

## 1) 데이터 및 시나리오

### 신용카드 고객정보 분석

> 삼성전자의 고객 마케팅에 활용하기 위하여 삼성 카드 측에서 비식별화된 고객 데이터를 협조 받아 통합 분석을 하기 위한 선제 분석을 하고자 한다.

※ 분석 수행 전 `기한 내 최소 지불 금액(MINIMUM_PAYMENTS)`의 결측 값(Null)을 각 컬럼의 평균값으로 대체하시오.

전처리 수행 결과를 `base` 객체로 지정하고 다음 문항에서 해당 객체를 기반으로 문제를 풀이하시오.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `card_cust.csv` | 1000 | 18 | UTF-8 |

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `CUST_ID` | int | 고객 ID |
| `BALANCE` | float | 연간 평균 잔고액 |
| `BALANCE_FREQUENCY` | float | 연중 잔고액 갱신 개월 수 비율 `(0~1 사이값)` |
| `PURCHASES` | float | 구매 총액 |
| `ONEOFF_PURCHASES` | float | 일시불 구매 총액 |
| `INSTALLMENTS_PURCHASES` | float | 할부 구매 총액 |
| `CASH_ADVANCE` | float | 현금서비스 구매 총액 |
| `PURCHASES_FREQUENCY` | float | 연중 구매 개월 수 비율 `(0~1 사이값)` |
| `ONEOFF_PURCHASES_FREQUENCY` | float | 연중 일시불 구매 개월 수 비율 `(0~1 사이값)` |
| `PURCHASES_INSTALLMENTS_FREQUENCY` | float | 연중 할부 구매 개월 수 비율 `(0~1 사이값)` |
| `CASH_ADVANCE_FREQUENCY` | float | 연중 현금서비스 구매 개월 수 비율 |
| `CASH_ADVANCE_TRX` | int | 현금 서비스 구매 횟수 |
| `PURCHASES_TRX` | int | 구매 횟수 |
| `CREDIT_LIMIT` | int | 신용카드 한도 |
| `PAYMENTS` | float | 지불 총액 |
| `MINIMUM_PAYMENTS` | float | 기한 내 최소 지불 금액 |
| `PRC_FULL_PAYMENT` | float | 연중 기한 내 전액 지불 개월 수 비율 `(0~1 사이값)` |
| `TENURE` | float | 신용카드 서비스 이용기간 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.preprocessing import StandardScaler` |
| `from sklearn.cluster import KMeans` |
| `from sklearn.metrics import silhouette_score` |
| `from sklearn.tree import DecisionTreeRegressor` |

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('../../dataset/card_cust.csv')
display(df.shape)
display(df.columns)
display(df.isna().any()) # 결측치 있는지 확인하기

(1000, 18)

Index(['CUST_ID', 'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES',
       'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE',
       'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
       'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY',
       'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS',
       'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE'],
      dtype='object')

In [9]:
# 전처리
df_base = df.copy()
display(df['MINIMUM_PAYMENTS'].isna().sum())

df_base['MINIMUM_PAYMENTS'] = df_base['MINIMUM_PAYMENTS'].fillna(df_base['MINIMUM_PAYMENTS'].mean())
display(df_base['MINIMUM_PAYMENTS'].isna().sum())

74

0

### Q01.

`base`를 사용하여 연간 평균 잔고액과 신용카드 서비스 이용기간 간의 관계를 파악하여, 추후 고객의 신용카드 한도 조정에 근거 자료로 활용하고자 한다.

연간 평균 잔고액(`BALANCE`)이 많을수록, 그리고 신용카드 서비스 이용기간(`TENURE`)이 길수록 신용카드 한도(`CREDIT_LIMIT`) 역시 높을 것으로 예상해볼 수 있다. 신용 카드 서비스 이용기간(`TENURE`) 별로 연간 평균 잔고액(`BALANCE`)과 신용카드 한도(`CREDIT_LIMIT`) 간 피어슨(Pearson) 상관 분석을 실시하고, 이 중 가장 큰 상관계수를 구하시오.

※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [15]:
df_q1 = df_base.copy()
display(df_q1['TENURE'].nunique(), df_q1['BALANCE'].nunique(),df_q1['CREDIT_LIMIT'].nunique())
df_q1_gb = df_q1.groupby('TENURE').apply(
    lambda df_balance_credit_limit : 
    df_balance_credit_limit['BALANCE'].corr(df_balance_credit_limit['CREDIT_LIMIT']) 
    )
display(df_q1_gb)
display(df_q1_gb.idxmax())
round(df_q1_gb.max(), 2)

7

997

93

TENURE
6.0     0.868056
7.0     0.948405
8.0     0.820696
9.0     0.085474
10.0    0.291482
11.0    0.380360
12.0    0.460833
dtype: float64

7.0

0.95


### Q02.

`base`를 사용하여 전략을 수립하기 위해 고객 세분화를 수행하고자 한다.  
일시불 구매 금액이 높은 고객군을 도출하기 위해 다음 단계에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

`고객 ID`를 제외한 모든 변수 17개에 대해 Z-score 표준화(Standardization) 한다.

#### 단계 2

표준화된 변수들에 대해 K-means 군집 분석을 수행한다.

이 때, 군집 수는 2~5개 중 K-means Silhouette를 통해 구한 최적의 K로 설정한다.

#### 단계 3

단계 2에서 도출한 각 군집 별로 `일시불 구매 총액`의 평균을 계산한다.

**군집 별 일시불 구매 총액(`ONEOFF_PURCHASES`)의 평균 중 가장 큰 값은 얼마인가?**

※ 정규화를 실시하지 않은 일시불 구매 총액 데이터를 기준으로 평균을 산출하시오.  
※ seed는 `1234`로 설정하시오.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [43]:
df_q2 = df_base.copy()
#단계 1
#D
X = df_q2.drop(columns=['CUST_ID']).copy()
#N
scaler = StandardScaler()
X_n = scaler.fit_transform(X) #array
display(X_n)
#단계 2
label = {}
sil = {}
for k in range(2,6) :
    model = KMeans(n_clusters = k,
                   random_state = 1234)
    y_pred = model.fit_predict(X_n)
    label[k] = y_pred
    sil[k] = silhouette_score(X_n,y_pred)

ser_sil = pd.Series(sil, name = 'sil')
K = ser_sil.idxmax()
display(K, ser_sil)
display("------------------")
df_label = pd.DataFrame(label[K])
display(df_label)

#단계 3
#X['cluster'] = df_label # series를 바로 붙인다.
X['cluster'] = label[K] # dict을 바로 붙인다.
X_gb = X.groupby('cluster')['ONEOFF_PURCHASES'].mean()
display(X_gb.idxmax(), X_gb.max())
round(X_gb.max(),2)


array([[-0.84876759, -0.41987944, -0.4419358 , ..., -0.44372465,
        -0.46554357,  0.28242902],
       [ 0.28279099,  0.01213096, -0.4690169 , ..., -0.08615941,
         0.33159169,  0.28242902],
       [ 0.0296341 ,  0.44414137, -0.24953794, ..., -0.25675456,
        -0.46554357,  0.28242902],
       ...,
       [-0.04555583,  0.44414137, -0.4690169 , ..., -0.1126367 ,
        -0.46554357,  0.28242902],
       [ 0.18482699,  0.44414137, -0.4233367 , ..., -0.07613978,
        -0.46554357,  0.28242902],
       [-0.34079285,  0.44414137, -0.20275918, ...,  0.00909944,
        -0.46554357,  0.28242902]])

2

2    0.307528
3    0.196361
4    0.207151
5    0.192741
Name: sil, dtype: float64

'------------------'

,0
0,0
1,0
2,0
3,0
4,0
...,...
995,0
996,0
997,0
998,0


1

3946.187525252525

3946.19

### Q03.

`base`를 사용하여 일시불 구매 총액(`ONEOFF_PURCHASES`) 예측 모델을 Target Marketing에 활용하고자 한다. 다음 단계에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

`고객 ID(CUST_ID)`가 4의 배수가 아닌 데이터를 Train Set으로, 4의 배수인 데이터를 Test Set으로 분할한다.

#### 단계 2

Train Set으로 아래 조건에 따라 의사결정나무 회귀모델을 학습한다.

- 독립 변수 총 16개: `고객 ID`, `일시불 구매 총액`을 제외한 모든 변수
- 종속 변수: `일시불 구매 총액`

#### 단계 3

생성된 모델을 Test Set에 적용하여 `일시불 구매 총액`을 예측한다.

**단계 3에서 얻은 예측 결과를 평가하기 위해, 아래 정의된 Measure B를 계산한 값은?**

$$
B = \left( \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2 \right)^{\frac{1}{2}}
$$

- $\hat{y_i}$: 예측값
- $y_i$: 실제값

※ seed는 `1234`로 설정하시오.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [42]:
df_q3 = df_base.copy()

#단계1
train = df_q3.loc[df_q3['CUST_ID']%4 != 0, :].copy()
test = df_q3.loc[df_q3['CUST_ID']%4 == 0, :].copy()
display(train.shape, test.shape)
#단계2
#D
X_train = train.drop(columns = ['CUST_ID', 'ONEOFF_PURCHASES']).copy()
X_test = test.drop(columns = ['CUST_ID', 'ONEOFF_PURCHASES']).copy()

y_train = train['ONEOFF_PURCHASES'].copy()
y_test = test['ONEOFF_PURCHASES'].copy()

#N
#M
model = DecisionTreeRegressor(random_state = 1234)
model.fit(X = X_train,y = y_train)
y_pred = model.predict(X_test)
#단계3
#E
B = ((y_pred - y_test)**2).mean()**0.5
round(B, 2)

(752, 18)

(248, 18)

1039.19